In [9]:
import pandas as pd
import numpy as np
import re
import string

# Impor NLTK (jalankan sekali jika belum di-download)
# import nltk
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

In [10]:
# Ambil stopwords bahasa Inggris
stop_words = set(stopwords.words('english'))
# Inisialisasi lemmatizer
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """Fungsi untuk membersihkan dan memproses teks."""
    if not isinstance(text, str):
        return ""
    
    # 1. Ubah ke huruf kecil
    text = text.lower()
    
    # 2. Hapus URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # 3. Hapus angka
    text = re.sub(r'\d+', '', text)
    
    # 4. Hapus tanda baca
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # 5. Tokenisasi (di notebook Anda, ini implisit, di sini kita buat eksplisit)
    tokens = text.split()
    
    # 6. Hapus stopwords dan lakukan lemmatization
    clean_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    
    # 7. Gabungkan kembali menjadi string
    return " ".join(clean_tokens)

In [12]:
try:
    # PERBAIKAN: Tambahkan header=None
    df_train = pd.read_csv("./twitter_training.csv", header=None)
    df_val = pd.read_csv("./twitter_validation.csv", header=None)
except FileNotFoundError:
    print("Pastikan file 'twitter_training.csv' dan 'twitter_validation.csv' berada di direktori yang sama.")
    # ... (sisanya bisa tetap sama)

# PERBAIKAN: Beri nama kolom SEBELUM di-drop
# Asumsi urutannya: ID, Platform, Sentimen, Teks
df_train.columns = ['ID', 'Borderlands', 'sentiment', 'text']
df_val.columns = ['ID', 'Borderlands', 'sentiment', 'text']

# SEKARANG baru kita hapus kolom yang tidak terpakai
# Ini seharusnya sudah berhasil
df_train = df_train.drop(columns=['ID', 'Borderlands'])
df_val = df_val.drop(columns=['ID', 'Borderlands'])

# Membersihkan data yang hilang (missing values)
df_train = df_train.dropna(subset=['text'])
df_val = df_val.dropna(subset=['text'])

# Membersihkan data duplikat
df_train = df_train.drop_duplicates()
df_val = df_val.drop_duplicates()

# Baris di bawah ini TIDAK DIPERLUKAN LAGI karena kita sudah 
# memberi nama 'sentiment' dan 'text' dengan benar di atas.
# HAPUS ATAU KOMENTARI BARIS INI:
# df_train.columns = ['sentiment', 'text']
# df_val.columns = ['sentiment', 'text']

print("Data berhasil dimuat dan dibersihkan.")
print("Contoh data latih setelah dibersihkan:")
print(df_train.head())

Data berhasil dimuat dan dibersihkan.
Contoh data latih setelah dibersihkan:
  sentiment                                               text
0  Positive  im getting on borderlands and i will murder yo...
1  Positive  I am coming to the borders and I will kill you...
2  Positive  im getting on borderlands and i will kill you ...
3  Positive  im coming on borderlands and i will murder you...
4  Positive  im getting on borderlands 2 and i will murder ...


In [13]:
df_train['clean_text'] = df_train['text'].apply(preprocess_text)

df_val['clean_text'] = df_val['text'].apply(preprocess_text)

In [14]:
vectorizer = TfidfVectorizer(max_features=5000)
label_encoder = LabelEncoder()
all_sentiments = pd.concat([df_train['sentiment'], df_val['sentiment']])
label_encoder.fit(all_sentiments)

# Terapkan label encoder
y_train = label_encoder.transform(df_train['sentiment'])
y_test = label_encoder.transform(df_val['sentiment'])

# Terapkan TF-IDF
X_train = vectorizer.fit_transform(df_train['clean_text'])
X_test = vectorizer.transform(df_val['clean_text'])

print(f"Bentuk X_train: {X_train.shape}")
print(f"Bentuk X_test: {X_test.shape}")

Bentuk X_train: (69769, 5000)
Bentuk X_test: (999, 5000)


In [15]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# Latih model
rf_model.fit(X_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [16]:
# Lakukan prediksi pada data uji (validasi)
y_pred_rf = rf_model.predict(X_test)

# Ubah kembali label prediksi dari angka ke teks (misal: 0 -> 'Negative')
y_pred_labels = label_encoder.inverse_transform(y_pred_rf)
y_test_labels = label_encoder.inverse_transform(y_test)

# Hitung akurasi
accuracy_rf = accuracy_score(y_test_labels, y_pred_labels)
print(f"\nAkurasi Random Forest: {accuracy_rf * 100:.2f}%")

# Tampilkan classification report
print("\nClassification Report untuk Random Forest:")
# Tentukan 'labels' untuk memastikan urutan yang benar
target_names = label_encoder.classes_
print(classification_report(y_test_labels, y_pred_labels, labels=target_names))


Akurasi Random Forest: 95.80%

Classification Report untuk Random Forest:
              precision    recall  f1-score   support

  Irrelevant       0.98      0.95      0.96       172
    Negative       0.96      0.96      0.96       266
     Neutral       0.96      0.95      0.96       285
    Positive       0.94      0.96      0.95       276

    accuracy                           0.96       999
   macro avg       0.96      0.96      0.96       999
weighted avg       0.96      0.96      0.96       999

